In [1]:
# Instalar PySpark en Google Colab
!pip install pyspark
!pip install polars pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812365 sha256=bf957af26a6e44f4096a0bbde117d511657bf42a2a6bb08fbdfdb59a3391b098
  Stored in directory: /root/.cache/pip/wheels/34/34/bd/03944534c44b677cd5859f248090daa9fb27b3c8f8e5f49574
Successfully built pyspark


In [21]:
import pandas as pd
import dask.dataframe as dd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
import polars as pl
import numpy as np
import time

# Crear una SparkSession
spark = SparkSession.builder.appName("heart").getOrCreate()

# Cargar el archivo CSV en un DataFrame de PySpark
file_path = "heart.csv"
spark_df = spark.read.csv(file_path, header=True, inferSchema=True)

# Obtener nombres de columnas numéricas
columnas_numericas = spark_df.columns

# Convertir DataFrame a formato de las diferentes librerías
# Dask
ddf = dd.from_pandas(spark_df.toPandas(), npartitions=100)

# Polars
pl_df = pl.DataFrame(spark_df.toPandas())

# Pandas y NumPy ya están en formato correcto
X_numeric = spark_df.toPandas()

# Diccionario para almacenar tiempos y resultados
times = {}

# Operaciones con PySpark para cada columna
start_time_spark = time.time()
mean_val_spark = {col: spark_df.select(F.mean(col)).collect()[0][0] for col in columnas_numericas}
median_val_spark = {col: spark_df.approxQuantile(col, [0.5], 0.01)[0] for col in columnas_numericas}
std_dev_spark = {col: spark_df.select(F.stddev(col)).collect()[0][0] for col in columnas_numericas}
times['PySpark'] = (time.time() - start_time_spark, mean_val_spark, median_val_spark, std_dev_spark)

# Operaciones con Dask para cada columna
start_time_dask = time.time()
mean_val_dask = {col: ddf[col].mean().compute() for col in columnas_numericas}
median_val_dask = {col: ddf[col].quantile(0.5).compute() for col in columnas_numericas}
std_dev_dask = {col: ddf[col].std().compute() for col in columnas_numericas}
times['Dask'] = (time.time() - start_time_dask, mean_val_dask, median_val_dask, std_dev_dask)

# Operaciones con Polars para cada columna
start_time_pl = time.time()
mean_val_pl = {col: pl_df.select(pl.col(col).mean()).to_numpy()[0][0] for col in columnas_numericas}
median_val_pl = {col: pl_df.select(pl.col(col).median()).to_numpy()[0][0] for col in columnas_numericas}
std_dev_pl = {col: pl_df.select(pl.col(col).std()).to_numpy()[0][0] for col in columnas_numericas}
times['Polars'] = (time.time() - start_time_pl, mean_val_pl, median_val_pl, std_dev_pl)

# Operaciones con Pandas para cada columna
start_time_pd = time.time()
mean_val_pd = {col: X_numeric[col].mean() for col in columnas_numericas}
median_val_pd = {col: X_numeric[col].median() for col in columnas_numericas}
std_dev_pd = {col: X_numeric[col].std() for col in columnas_numericas}
times['Pandas'] = (time.time() - start_time_pd, mean_val_pd, median_val_pd, std_dev_pd)

# Operaciones con NumPy para cada columna
start_time_np = time.time()
mean_val_np = {col: np.mean(X_numeric[col]) for col in columnas_numericas}
median_val_np = {col: np.median(X_numeric[col]) for col in columnas_numericas}
std_dev_np = {col: np.std(X_numeric[col]) for col in columnas_numericas}
times['NumPy'] = (time.time() - start_time_np, mean_val_np, median_val_np, std_dev_np)

# Calcular la herramienta más rápida
fastest_tool = min(times, key=lambda x: times[x][0])
fastest_time = times[fastest_tool][0]

# Calcular cuántas veces es más rápida la herramienta más rápida que las demás
speedups = {tool: fastest_time / time_taken[0] for tool, time_taken in times.items()}

# Imprimir resultados
print("Tiempos de ejecución y estadísticas para todas las columnas:")
for tool, (time_taken, mean_vals, median_vals, std_devs) in times.items():
    print(f"{tool}: Tiempo: {time_taken:.4f} segundos")
    for col in columnas_numericas:
        print(f"Columna: {col}, Media: {mean_vals[col]:.6f}, Mediana: {median_vals[col]:.6f}, Desv. Est.: {std_devs[col]:.6f}")

print(f"\nLa herramienta más rápida fue {fastest_tool} con {fastest_time:.4f} segundos.")


Tiempos de ejecución y estadísticas para todas las columnas:
PySpark: Tiempo: 14.0670 segundos
Columna: age, Media: 54.366337, Mediana: 55.000000, Desv. Est.: 9.082101
Columna: sex, Media: 0.683168, Mediana: 1.000000, Desv. Est.: 0.466011
Columna: cp, Media: 0.966997, Mediana: 1.000000, Desv. Est.: 1.032052
Columna: trtbps, Media: 131.623762, Mediana: 130.000000, Desv. Est.: 17.538143
Columna: chol, Media: 246.264026, Mediana: 240.000000, Desv. Est.: 51.830751
Columna: fbs, Media: 0.148515, Mediana: 0.000000, Desv. Est.: 0.356198
Columna: restecg, Media: 0.528053, Mediana: 1.000000, Desv. Est.: 0.525860
Columna: thalachh, Media: 149.646865, Mediana: 152.000000, Desv. Est.: 22.905161
Columna: exng, Media: 0.326733, Mediana: 0.000000, Desv. Est.: 0.469794
Columna: oldpeak, Media: 1.039604, Mediana: 0.800000, Desv. Est.: 1.161075
Columna: slp, Media: 1.399340, Mediana: 1.000000, Desv. Est.: 0.616226
Columna: caa, Media: 0.729373, Mediana: 0.000000, Desv. Est.: 1.022606
Columna: thall, Med